# Phase 7.3: Production Edge-Case Audit — Silent Active Gateways

This notebook audits the operational behavior of active gateways with zero telemetry records
during the recent 7-day scoring window $[T-7\text{d}, T)$ across the eight challenge-scored Mondays.

**Key Questions:**
1. How many active gateways are completely silent in the recent 7-day scoring window?
2. How many positive-score vs. zero-score gateways exist in each week?
3. Does retaining silent gateways with score 0 affect the submitted Top-15 recommendations?
4. What is the evidence-backed policy recommendation (Option A vs. Option B vs. Option C)?

In [ ]:
import sys
sys.path.insert(0, '../src')

import datetime as dt
from pathlib import Path
import pandas as pd
import numpy as np

from nexora.data_loader import DataLoader, normalize_gateway_id
from nexora.backtesting.strategies import Baseline3SigmaStrategy
import baseline_3sigma

data_dir = Path('../data')
dl = DataLoader(data_dir)
master = dl.load_master()
telemetry = dl.load_telemetry()

SCORED_WEEKS = [dt.date(2026, 2, 2) + dt.timedelta(days=7 * i) for i in range(8)]
print(f'Loaded {len(master)} master gateways and {len(telemetry):,} deduplicated telemetry rows.')

In [ ]:
audit_rows = []

for monday in SCORED_WEEKS:
    t = dt.datetime.combine(monday, dt.time.min, tzinfo=dt.timezone.utc)
    t_28 = t - dt.timedelta(days=28)
    t_7 = t - dt.timedelta(days=7)
    
    # Eligible active universe
    eligible = master[
        (master['installed_on_dt'] <= t) & 
        ((master['decommissioned_on_dt'] > t) | master['decommissioned_on_dt'].isna())
    ].copy()
    eligible_ids = set(eligible['gateway_id'])
    
    # Telemetry windows
    t_base = telemetry[(telemetry['ts'] >= t_28) & (telemetry['ts'] < t)]
    t_recent = telemetry[(telemetry['ts'] >= t_7) & (telemetry['ts'] < t)]
    
    present_recent_ids = set(t_recent['gateway_id']) & eligible_ids
    silent_ids = eligible_ids - present_recent_ids
    
    present_base_ids = set(t_base['gateway_id']) & eligible_ids
    silent_with_28d = silent_ids & present_base_ids
    silent_no_28d = silent_ids - present_base_ids
    
    # Run baseline ranking
    ranked = baseline_3sigma.rank_week(telemetry, monday)
    ranked['gateway_id'] = ranked['gateway_id'].apply(normalize_gateway_id)
    ranked_eligible = ranked[ranked['gateway_id'].isin(eligible_ids)].copy()
    
    pos_scores = int((ranked_eligible['flagged_hours'] > 0).sum())
    zero_scores = len(eligible_ids) - pos_scores
    
    # Option B Top 15
    missing_ids = eligible_ids - set(ranked_eligible['gateway_id'])
    if missing_ids:
        missing_df = pd.DataFrame({
            'gateway_id': sorted(missing_ids),
            'flagged_hours': 0,
            'worst_metric': 'no_telemetry',
        })
        ranked_b = pd.concat([ranked_eligible, missing_df], ignore_index=True)
    else:
        ranked_b = ranked_eligible.copy()
    
    ranked_b = ranked_b.sort_values(
        by=['flagged_hours', 'gateway_id'], ascending=[False, True]
    ).reset_index(drop=True)
    
    top15_ids = set(ranked_b.head(15)['gateway_id'])
    silent_in_top15 = len(silent_ids & top15_ids) > 0
    
    audit_rows.append({
        'Week': str(monday),
        'Eligible': len(eligible_ids),
        'Telemetry-present': len(present_recent_ids),
        'Silent': len(silent_ids),
        'Silent with 28d history': len(silent_with_28d),
        'Silent with no 28d history': len(silent_no_28d),
        'Positive-score gateways': pos_scores,
        'Zero-score gateways': zero_scores,
        'Silent enters Top-15?': 'Yes' if silent_in_top15 else 'No',
        'Silent Gateway IDs': sorted(list(silent_ids))
    })

audit_df = pd.DataFrame(audit_rows)
audit_df

In [ ]:
# Verify identity of Top-15 recommendations between Option A and Option B
strategy = Baseline3SigmaStrategy(data_dir)

for monday in SCORED_WEEKS:
    t = dt.datetime.combine(monday, dt.time.min, tzinfo=dt.timezone.utc)
    eligible = master[
        (master['installed_on_dt'] <= t) & 
        ((master['decommissioned_on_dt'] > t) | master['decommissioned_on_dt'].isna())
    ].copy()
    
    # Option B (retaining silent gateways with score 0)
    ranked_b = strategy.rank(eligible, monday)
    top15_b = ranked_b.head(15)['gateway_id'].tolist()
    
    # Option A (omitting silent gateways)
    ranked_a = baseline_3sigma.rank_week(telemetry, monday)
    ranked_a['gateway_id'] = ranked_a['gateway_id'].apply(normalize_gateway_id)
    ranked_a = ranked_a[ranked_a['gateway_id'].isin(eligible['gateway_id'])].sort_values(
        by=['flagged_hours', 'gateway_id'], ascending=[False, True]
    ).reset_index(drop=True)
    top15_a = ranked_a.head(15)['gateway_id'].tolist()
    
    assert top15_a == top15_b, f'Mismatch on {monday}!'
    print(f'{monday}: Option A and Option B Top-15 are 100% IDENTICAL.')

print('\nVerification confirmed: Silent-gateway treatment has ZERO impact on Top-15 recommendations.')